# Ghost in the Aether — Populate Lakehouse

Creates and populates the three dimension tables that form the static data layer for the murder mystery game:
- `dimevidence` — Physical and digital evidence items
- `dimlocation` — Locations on the Aetherium Estate
- `dimperson` — Characters (victim + suspects) with bios and images

In [ ]:
%%sql
-- ============================================================
-- dimevidence: Physical and digital evidence items
-- ============================================================
CREATE TABLE IF NOT EXISTS dimevidence (
  `EvidenceID` int,
  `EvidenceName` string,
  `Description` string,
  `EvidenceType` string,
  `InitialLocationID` int,
  `RevealedByDialogueID` int
);

INSERT OVERWRITE dimevidence VALUES
  (201, 'Partial Email Draft', 'An unsent email on Evelyn''s laptop to a tech journalist, detailing Julian''s theft of her code.', 'Digital', 103, NULL),
  (202, 'Suspicious Server Log', 'Aether''s security log shows Evelyn''s keycard was used to access the server room at 2:15 AM.', 'Digital', 102, NULL),
  (203, 'Threatening Text Messages', 'A series of angry texts from Marcus to Julian''s phone, sent the night of the murder.', 'Digital', 101, NULL),
  (204, 'Shattered Picture Frame', 'A photo of Marcus and Julian in happier times, now in a broken frame in Marcus''s suite.', 'Physical', 104, NULL),
  (205, 'Disguised Audio Recorder', 'A pen in Anya''s bag is a sophisticated audio recorder, containing a heated argument with Julian.', 'Physical', 103, NULL),
  (206, 'Muddy Shoes', 'Traces of mud on Anya''s shoes match the soil from the secluded path behind the victim''s study.', 'Physical', 103, NULL),
  (207, 'Professor''s Journal', 'Dr. Finch''s personal journal contains entries calling Aether "the atomic bomb of our generation" and stating that "drastic measures are required".', 'Physical', 106, NULL),
  (208, 'Biochemistry Degree', 'University records show Dr. Finch holds an advanced degree in biochemistry.', 'Digital', NULL, NULL),
  (209, 'Vintage Whiskey Bottle', 'The empty bottle of rare whiskey on Julian''s desk. Alistair mentioned gifting it to him.', 'Physical', 101, NULL);

In [ ]:
%%sql
-- ============================================================
-- dimlocation: Locations on the Aetherium Estate
-- ============================================================
CREATE TABLE IF NOT EXISTS dimlocation (
  `LocationID` int,
  `LocationName` string,
  `Description` string,
  `IsInitiallyLocked` boolean
);

INSERT OVERWRITE dimlocation VALUES
  (101, 'Julian''s Study', 'A pristine, minimalist office with a large oak desk and a single glass of whiskey. The air is cold.', True),
  (102, 'Server Room', 'A chilled room filled with humming server racks that house Aether. The heart of the island.', True),
  (103, 'Evelyn''s Suite', 'A spartan room, tidy and impersonal, dominated by a high-end laptop covered in coding stickers.', False),
  (104, 'Marcus''s Suite', 'An opulent suite in disarray. A shattered picture frame lies on the floor.', False),
  (105, 'Back Path', 'A muddy, secluded path leading from the gardens to the rear of Julian''s study.', False),
  (106, 'Dr. Finch''s Suite', 'The professor''s personal quarters, lined with books and smelling of pipe tobacco.', False);

In [ ]:
%%sql
-- ============================================================
-- dimperson: Characters — victim and suspects
-- ============================================================
CREATE TABLE IF NOT EXISTS dimperson (
  `PersonID` int,
  `PersonName` string,
  `PersonRole` string,
  `Motive` string,
  `Secret` string,
  `Bio` string,
  `ImageURL` string,
  `ExtendedBio` string
);

In [ ]:
# Insert dimperson rows via PySpark to handle large base64 image strings cleanly
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("PersonID", IntegerType(), False),
    StructField("PersonName", StringType(), False),
    StructField("PersonRole", StringType(), False),
    StructField("Motive", StringType(), True),
    StructField("Secret", StringType(), True),
    StructField("Bio", StringType(), True),
    StructField("ImageURL", StringType(), True),
    StructField("ExtendedBio", StringType(), True),
])

persons = [
    (1, "Julian Croft", "Victim", None, None,
     "A ruthless, brilliant, and manipulative tech billionaire. Creator of the Aether AI.",
     None,  # Image loaded separately
     "To the world, Julian Croft wasn't a man; he was an event. A self-made titan whose myth was as carefully engineered as the code that made him billions."),
    (2, "Evelyn Reed", "Suspect",
     "Revenge: Julian stole her work, patented it under his name, and pushed her out of the company.",
     "She has a custom-made USB drive with a virus designed to wipe Aether's core programming.",
     "A fiercely intelligent but socially awkward programmer. The true architect of the Aether AI's core code.",
     None,
     "Evelyn Reed moves through the opulent halls of the Aetherium Estate like a ghost, her gaze fixed on her tablet screen as if the physical world is a low-resolution distraction."),
    (3, "Marcus Thorne", "Suspect",
     "Betrayal: Julian cruelly ended their romantic and business partnership, leaving him with nothing.",
     "He is massively in debt from a series of bad private investments and faces financial ruin without the company.",
     "The charismatic and handsome Chief Operating Officer of Aetherium. He was the public face of the company.",
     None,
     "To the world, Marcus Thorne is the handsome, charismatic face of Aetherium Inc.—the man who sold Julian Croft's vision with a killer smile and a perfectly tailored suit."),
    (4, "Anya Sharma", "Suspect",
     "Survival: The launch of Aether would make her company's technology obsolete, guaranteeing its collapse.",
     "She has a contact in Aetherium's chemistry division who could have given her access to rare chemicals.",
     "The sharp, cold, and calculating CEO of a competing tech firm, Nexus Dynamics.",
     None,
     "Anya Sharma, CEO of the rival firm Nexus Dynamics, exudes an aura of predatory calm. Every handshake is a calculation, every word a strategic move."),
    (5, "Dr. Alistair Finch", "Suspect",
     "Ideology: He believes Aether is too dangerous for humanity and discovered Julian's plan to sell it to military contractors.",
     "He has a terminal heart condition with only months to live and is not afraid of the consequences.",
     "A respected, older academic specializing in AI ethics. He was Julian's former university professor and mentor.",
     None,
     "Dr. Alistair Finch is a man of quiet dignity, his academic reputation a stark contrast to the cutthroat world of his former student, Julian Croft."),
]

df = spark.createDataFrame(persons, schema=schema)
df.write.mode("overwrite").format("delta").saveAsTable("dimperson")
print(f"dimperson: {df.count()} rows written")

In [ ]:
%%sql
-- Verify row counts
SELECT 'dimevidence' AS tableName, COUNT(*) AS rowCount FROM dimevidence
UNION ALL
SELECT 'dimlocation', COUNT(*) FROM dimlocation
UNION ALL
SELECT 'dimperson', COUNT(*) FROM dimperson;